# 01. Clean the data (Task 1)

**Project:** TMS, the data scientist of a travel agency that books flights from New York for its clients. Data: NYC flights 2023 (every flight that left JFK, LGA or EWR), plus weather, planes, airports and airlines tables.

## The three business questions

1. **Q1 (prediction 1).** Which flights will leave more than 15 minutes late, or be cancelled, so the agency can warn clients two hours before departure?
2. **Q2 (tasks 2, 3, 4).** Which airlines, airports, hours and months are the most and the least reliable, and how much does the weather explain?
3. **Q3 (prediction 2).** Which flights will be cancelled, so clients can be rebooked before it happens?

## What this notebook does

Check every column of the five files before changing anything, list the problems with counts, clean the data into **new** files (the original files in `data/raw/` are never changed), and end with a data report table: problem, rows affected, decision, reason.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from utils import *

set_seed()                      # SEED = 42, same numbers on every run
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

## 1. Load the original files

The files are read exactly as they are. Text columns are read as text on purpose (for example `time_hour`), so nothing is guessed by the reader. The column names of each file are compared with the COLUMNS sheet of the course workbook, and the row counts with the PROJECTS sheet (flights: 435,352 rows).

In [2]:
text = {"tailnum": "str", "carrier": "str", "origin": "str", "dest": "str", "time_hour": "str"}
flights_raw  = pd.read_csv(RAW / "flights.csv",  dtype=text)
weather_raw  = pd.read_csv(RAW / "weather.csv",  dtype={"origin": "str", "time_hour": "str"})
planes_raw   = pd.read_csv(RAW / "planes.csv")
airports_raw = pd.read_csv(RAW / "airports.csv")
airlines_raw = pd.read_csv(RAW / "airlines.csv")

# Column lists written in the COLUMNS sheet of ML_week_tasks.xlsx
expected_cols = {
    "flights": ["year","month","day","dep_time","sched_dep_time","dep_delay","arr_time","sched_arr_time",
                "arr_delay","carrier","flight","tailnum","origin","dest","air_time","distance","hour","minute","time_hour"],
    "weather": ["origin","year","month","day","hour","temp","dewp","humid","wind_dir","wind_speed","wind_gust",
                "precip","pressure","visib","time_hour"],
    "planes": ["tailnum","year","type","manufacturer","model","engines","seats","speed","engine"],
    "airports": ["faa","name","lat","lon","alt","tz","dst","tzone"],
    "airlines": ["carrier","name"],
}
raw = {"flights": flights_raw, "weather": weather_raw, "planes": planes_raw,
       "airports": airports_raw, "airlines": airlines_raw}
for name, df in raw.items():
    assert sorted(df.columns) == sorted(expected_cols[name]), f"{name}: columns differ from the COLUMNS sheet"
assert flights_raw.shape == (435_352, 19), flights_raw.shape        # PROJECTS sheet: 435,352 rows
print({k: v.shape for k, v in raw.items()})

{'flights': (435352, 19), 'weather': (26207, 15), 'planes': (4840, 9), 'airports': (1255, 8), 'airlines': (14, 2)}


## 2. Check every column before changing anything

For every column of every file: its type, how many values are missing, how many different values it has, and its smallest and largest value. (This is a summary of the columns, not a dump of the data.)

In [3]:
print("flights.csv"); display(profile(flights_raw))

flights.csv


,type,missing,distinct,min,max
column,,,,,
year,int64,0,1,2023,2023
month,int64,0,12,1,12
day,int64,0,31,1,31
dep_time,float64,10738,1365,1.0,2400.0
sched_dep_time,int64,0,1060,500,2359
dep_delay,float64,10738,946,-50.0,1813.0
arr_time,float64,11453,1433,1.0,2400.0
sched_arr_time,int64,0,1200,1,2359
arr_delay,float64,12534,1019,-97.0,1812.0


In [4]:
print("weather.csv"); display(profile(weather_raw))

weather.csv


,type,missing,distinct,min,max
column,,,,,
origin,str,0,3,EWR,LGA
year,int64,0,1,2023,2023
month,int64,0,12,1,12
day,int64,0,31,1,31
hour,int64,0,24,0,23
temp,float64,2,123,5.0,96.0
dewp,float64,2,132,-17.0,77.0
humid,float64,2,2989,10.54,100.0
wind_dir,float64,1225,37,0.0,360.0


In [5]:
print("planes.csv"); display(profile(planes_raw))
print("airports.csv"); display(profile(airports_raw))
print("airlines.csv"); display(profile(airlines_raw))

planes.csv


,type,missing,distinct,min,max
column,,,,,
tailnum,str,0,4840,N101DQ,N999JQ
year,float64,89,34,1990.0,2023.0
type,str,0,3,Fixed wing multi engine,Rotorcraft
manufacturer,str,0,12,AIRBUS,EUROCOPTER DEUTSCHLAND GMBH
model,str,0,92,717-200,ERJ 190-100 IGW
engines,int64,0,3,1,3
seats,int64,0,44,4,563
speed,int64,0,1,0,0
engine,str,0,4,Reciprocating,Turbo-shaft


airports.csv


,type,missing,distinct,min,max
column,,,,,
faa,str,0,1255,AAF,ZZV
name,str,0,1252,Aberdeen Regional Airport,Zephyrhills Municipal Airport
lat,float64,0,1255,17.995278,71.285402
lon,float64,0,1253,-176.645996,174.113998
alt,float64,0,860,-115.0,9070.0
tz,float64,48,9,-10.0,8.0
dst,str,48,4,A,U
tzone,str,119,11,America/Adak,Pacific/Honolulu


airlines.csv


,type,missing,distinct,min,max
column,,,,,
carrier,str,0,14,9E,YX
name,str,0,14,Alaska Airlines Inc.,United Air Lines Inc.


**What the tables show, in plain words**

* In `flights.csv` the columns `dep_time` and `dep_delay` have exactly the same number of missing values. These are the cancelled flights.
* `tailnum` is missing for some flights. Section 3 checks whether those are the cancelled ones.
* `hour`, `minute`, `distance` and `air_time` are stored as decimals only because some values are missing; they are whole numbers.
* `planes.csv`: the column `speed` is 0 for every plane, so it carries no information.
* `weather.csv` has fewer rows than 3 airports x 365 days x 24 hours = 26,280. Section 3 finds which hours are missing.

## 3. Look for problems

Each check prints a count. These counts feed the data report at the end.

In [6]:
f = flights_raw
counts = {}

# 3.1 duplicate rows
counts["duplicate rows (flights)"] = int(f.duplicated().sum())
counts["duplicate rows (weather)"] = int(weather_raw.duplicated().sum())
counts["duplicate (origin, time_hour) in weather"] = int(weather_raw.duplicated(["origin", "time_hour"]).sum())
counts["duplicate tailnum in planes"] = int(planes_raw.tailnum.duplicated().sum())

# 3.2 cancelled flights and what is missing together with them
cancelled = f.dep_time.isna()
counts["cancelled flights (dep_time missing)"] = int(cancelled.sum())
counts["dep_time missing but dep_delay present"] = int((cancelled & f.dep_delay.notna()).sum())
counts["tailnum missing"] = int(f.tailnum.isna().sum())
counts["tailnum missing AND flight not cancelled"] = int((f.tailnum.isna() & ~cancelled).sum())
counts["cancelled flights that still have a tailnum"] = int((cancelled & f.tailnum.notna()).sum())
counts["departed but arr_time missing"] = int((f.dep_time.notna() & f.arr_time.isna()).sum())
counts["arr_time present but arr_delay missing"] = int((f.arr_time.notna() & f.arr_delay.isna()).sum())

# 3.3 times stored as numbers HHMM
for c in ["dep_time", "sched_dep_time", "arr_time", "sched_arr_time"]:
    v = f[c].dropna()
    counts[f"{c}: minutes part >= 60"] = int(((v % 100) >= 60).sum())
counts["dep_time == 2400"] = int((f.dep_time == 2400).sum())
counts["arr_time == 2400"] = int((f.arr_time == 2400).sum())

# 3.4 impossible or extreme values
counts["negative air_time or distance"] = int(((f.air_time < 0) | (f.distance <= 0)).sum())
counts["dep_delay below -30 minutes"] = int((f.dep_delay < -30).sum())
counts["dep_delay above 720 minutes (12 h)"] = int((f.dep_delay > 720).sum())
speed = f.distance / (f.air_time / 60)
counts["implied speed below 150 mph"] = int((speed < 150).sum())
counts["implied speed above 700 mph"] = int((speed > 700).sum())

# 3.5 the same value written in different ways (case, spaces)
for c in ["carrier", "origin", "dest", "tailnum"]:
    v = f[c].dropna()
    counts[f"{c}: variants by case/spaces"] = int((v != v.str.strip().str.upper()).sum())

# 3.6 dates and times stored as text or numbers
counts["time_hour stored as text (all rows)"] = int(f.time_hour.map(type).eq(str).sum())
counts["hour != sched_dep_time // 100"] = int((f.hour != f.sched_dep_time // 100).sum())
counts["minute != sched_dep_time % 100"] = int((f.minute != f.sched_dep_time % 100).sum())

pd.Series(counts, name="rows").to_frame()

,rows
duplicate rows (flights),0
duplicate rows (weather),0
"duplicate (origin, time_hour) in weather",0
duplicate tailnum in planes,0
cancelled flights (dep_time missing),10738
dep_time missing but dep_delay present,0
tailnum missing,1913
tailnum missing AND flight not cancelled,0
cancelled flights that still have a tailnum,8825
departed but arr_time missing,715


**Reading the counts**

* No duplicate rows and no spelling variants.
* Every flight with a missing `tailnum` is cancelled (the line "tailnum missing AND flight not cancelled" must be 0). This matters a lot later: a model must never learn "tail missing means cancelled" (see the plan, trap T2).
* `dep_delay` above 720 minutes are real flights that left the next day, not errors. The recomputed delay from the clock times differs only for those flights, because the clock times have no date.
* A time written 2400 means midnight; it is turned into 0 in a new column.
* About 70 values of speed look too slow or too fast for a plane. They are flagged, not removed: the flight happened, only `air_time` may be wrong.

In [7]:
# Weather: which hours are missing, and where are values missing?
expected_hours = pd.date_range("2023-01-01", "2023-12-31 23:00", freq="h")
w_time = pd.to_datetime(weather_raw.time_hour)
missing_hours = {}
for o in ["EWR", "JFK", "LGA"]:
    have = set(w_time[weather_raw.origin == o])
    gone = sorted(set(expected_hours) - have)
    missing_hours[o] = gone
    print(o, "rows:", len(have), "| hours missing:", len(gone), "| first:", gone[0], "| last:", gone[-1])

print()
print("Missing values per weather column:")
print(weather_raw.isna().sum()[lambda s: s > 0].to_dict())
print("Calm wind (speed 0) recorded with direction 0:", int(((weather_raw.wind_speed == 0) & (weather_raw.wind_dir == 0)).sum()))
print("wind_speed and wind_gust missing together:", bool((weather_raw.wind_speed.isna() == weather_raw.wind_gust.isna()).all()))

EWR rows: 8736 | hours missing: 24 | first: 2023-12-31 00:00:00 | last: 2023-12-31 23:00:00
JFK rows: 8736 | hours missing: 24 | first: 2023-12-31 00:00:00 | last: 2023-12-31 23:00:00
LGA rows: 8735 | hours missing: 25 | first: 2023-05-02 05:00:00 | last: 2023-12-31 23:00:00

Missing values per weather column:
{'temp': 2, 'dewp': 2, 'humid': 2, 'wind_dir': 1225, 'wind_speed': 1041, 'wind_gust': 1041, 'pressure': 10, 'visib': 23}
Calm wind (speed 0) recorded with direction 0: 1255
wind_speed and wind_gust missing together: True


**Weather time zone (important).** The weather file is in **UTC** (the nycflights23 documentation says so, and the data agrees: the coldest hour is about 9 and the warmest about 18 to 19 in every month, which only fits UTC). The flights file is in **New York local time**. The next cell shows the evidence. A join of the two tables on the raw `time_hour` text would pair each flight with the weather of 4 or 5 hours earlier or later, so the cleaning converts the flights' time to UTC before joining (decision D4 in the plan).

In [8]:
wt = weather_raw.assign(t=w_time, hr=w_time.dt.hour, mo=w_time.dt.month)
by_hour = wt.groupby("hr").temp.mean()
print("July mean temperature by hour in weather.csv: coldest hour =", wt[wt.mo == 7].groupby("hr").temp.mean().idxmin(),
      "| warmest hour =", wt[wt.mo == 7].groupby("hr").temp.mean().idxmax(), "(local New York would be about 5 and 15)")
fh = flights_raw.hour.value_counts().sort_index()
print("Busiest scheduled hour in flights.csv:", int(fh.idxmax()), "| first hour:", int(fh.index.min()), "| last hour:", int(fh.index.max()), "(local time, as the course sheet says)")

July mean temperature by hour in weather.csv: coldest hour = 9 | warmest hour = 19 (local New York would be about 5 and 15)
Busiest scheduled hour in flights.csv: 9 | first hour: 5 | last hour: 23 (local time, as the course sheet says)


## 4. What the data does not tell you

* **Where:** only flights that **left** New York's three airports (JFK, LGA, EWR). Nothing about flights arriving in New York or flying between other cities.
* **When:** one year, 2023. Another year can look different, so patterns found here may not repeat.
* **Who is missing:** no passengers, no prices, no tickets. The data describes flights, not clients.
* **No reasons:** nothing says why a flight was cancelled or delayed, and there is no time at which a cancellation became known. A feature can therefore not use "was an earlier flight cancelled" safely.
* **Diversions:** 715 flights departed but have no arrival information and 1,081 more have an arrival time but no arrival delay. The data dictionary does not explain them; they are not called "diverted" here.
* **Time zones:** departures are New York time, arrivals are the local time of the destination, and the weather table is UTC.
* **Weather:** it is observed weather at the airport, not a forecast. It has gaps (the whole of Dec 31, some single hours).
* **Planes:** 277 tail numbers used by flights are not in the planes table, and every cancelled flight with a missing tail number has no plane information at all.
* **How it was collected:** described only on the nycflights23 package pages (the weather page names ASOS data from the Iowa Environmental Mesonet as its source). The flights page should be read and quoted by the author before the final report.

## 5. Clean the data

For every problem the decision is one of three: **fix** it, **remove** the rows, or **keep** the rows and add a **flag** column. No flight is removed: every row is a real flight, and removing rows (for example the cancelled ones) would bias the monthly numbers. Everything is saved to new files in `data/processed/`.

### 5.1 Flights

In [9]:
fl = flights_raw.copy()
fl.insert(0, "flight_row_id", np.arange(len(fl)))        # position in the original file, used to align files later

# Whole numbers stored as decimals -> nullable integers (missing stays missing)
int_cols = ["dep_time", "sched_dep_time", "dep_delay", "arr_time", "sched_arr_time", "arr_delay",
            "air_time", "distance", "hour", "minute"]
for c in int_cols:
    assert (fl[c].dropna() % 1 == 0).all(), c
    fl[c] = fl[c].astype("Int32")

# Dates: time_hour is New York local time written as text. Make it a real date-time with its time zone.
local = pd.to_datetime(fl.time_hour).dt.tz_localize("America/New_York", ambiguous="raise", nonexistent="raise")
fl["time_hour_naive"] = local.dt.tz_localize(None)       # the raw text as a date-time (only for the 'literal join' comparison)
fl["time_hour"] = local
fl["time_hour_utc"] = local.dt.tz_convert("UTC")
fl["sched_dt"] = local + pd.to_timedelta(fl.minute.astype("int64"), unit="m")           # scheduled departure, local
fl["actual_dep_dt"] = fl.sched_dt + pd.to_timedelta(fl.dep_delay.astype("float64"), unit="m")  # actual departure (NaT if cancelled)

# Midnight written as 2400 -> 0 (new columns, the originals stay)
fl["dep_time_fixed"] = fl.dep_time.where(fl.dep_time != 2400, 0)
fl["arr_time_fixed"] = fl.arr_time.where(fl.arr_time != 2400, 0)

# Targets (decisions D1 and D3 of the plan)
fl["cancelled"] = fl.dep_time.isna().astype("int8")
fl["late"] = (fl.dep_delay.gt(15).fillna(False) | fl.dep_time.isna()).astype("int8")     # exactly 15 minutes is NOT late

# Flags: they describe data quality for reports. None of them may be used as a model feature.
planes_tails = set(planes_raw.tailnum)
implied_speed = fl.distance.astype("float64") / (fl.air_time.astype("float64") / 60)
fl["is_cancelled"] = fl.cancelled.astype(bool)
fl["tail_missing"] = fl.tailnum.isna()
fl["tail_not_in_planes"] = fl.tailnum.notna() & ~fl.tailnum.isin(planes_tails)
fl["arrival_missing_but_departed"] = fl.dep_time.notna() & fl.arr_time.isna()
fl["arr_delay_missing_with_arr_time"] = fl.arr_time.notna() & fl.arr_delay.isna()
fl["overnight_delay"] = fl.dep_delay.gt(720).fillna(False).astype(bool)
fl["speed_flag"] = ((implied_speed < 150) | (implied_speed > 700)).fillna(False).astype(bool)

assert len(fl) == 435_352
fl[["flight_row_id", "origin", "dest", "sched_dt", "dep_delay", "late", "cancelled"]].head()

,flight_row_id,origin,dest,sched_dt,dep_delay,late,cancelled
0,0,EWR,SMF,2023-01-01 20:38:00-05:00,203,1,0
1,1,JFK,ATL,2023-01-01 23:00:00-05:00,78,1,0
2,2,JFK,BQN,2023-01-01 23:44:00-05:00,47,1,0
3,3,JFK,CHS,2023-01-01 21:40:00-05:00,173,1,0
4,4,EWR,DTW,2023-01-01 20:48:00-05:00,228,1,0


### 5.2 Weather, planes, airports, airlines

In [10]:
# Weather: the time is UTC. Keep one clear time column, drop the split copies (year, month, day, hour) to avoid confusion.
we = weather_raw.copy()
we["time_hour_utc"] = pd.to_datetime(we.time_hour).dt.tz_localize("UTC")
we["wind_calm"] = (we.wind_speed == 0)                  # direction 0 is written for calm wind: not a real direction
we = we.drop(columns=["year", "month", "day", "hour", "time_hour"])
assert not we.duplicated(["origin", "time_hour_utc"]).any()      # the missing hours are NOT filled in (no inventing data)

# Planes: drop 'speed' (0 for every plane). Year stays (89 missing).
pl = planes_raw.drop(columns=["speed"]).copy()
assert pl.tailnum.is_unique

# Airports and airlines: keep as they are
ap = airports_raw.copy()
al = airlines_raw.copy()

# The airports we really use must have a time zone
used = set(fl.origin) | set(fl.dest)
print("airports used by flights:", len(used), "| of them with tz missing:", int(ap[ap.faa.isin(used)].tz.isna().sum()))

airports used by flights: 121 | of them with tz missing: 0


### 5.3 Check that the tables join as expected

Every join must keep exactly 435,352 flights (no row lost, none doubled). The counts of flights without a match are part of the data report.

In [11]:
n = len(fl)
joins = {}

m = fl.merge(al, on="carrier", how="left", validate="m:1")
joins["airlines on carrier: flights without match"] = int(m["name"].isna().sum()); assert len(m) == n

m = fl.merge(pl[["tailnum", "year"]].rename(columns={"year": "plane_year"}), on="tailnum", how="left", validate="m:1")
joins["planes on tailnum: flights with a tail but no plane row"] = int(fl.tail_not_in_planes.sum())
joins["planes on tailnum: distinct tails not in planes.csv"] = int(fl.loc[fl.tail_not_in_planes, "tailnum"].nunique())
assert len(m) == n

m = fl.merge(ap[["faa", "tz"]], left_on="dest", right_on="faa", how="left", validate="m:1")
joins["airports on dest: flights without match"] = int(m.faa.isna().sum()); assert len(m) == n

# Weather: correct join (flight time converted to UTC) versus the literal join of the course sheet
wcols = ["origin", "time_hour_utc", "temp"]
m = fl.merge(we[wcols], on=["origin", "time_hour_utc"], how="left", validate="m:1")
joins["weather, UTC-aware join: flights without weather temp"] = int(m.temp.isna().sum()); assert len(m) == n
we_lit = weather_raw.assign(time_hour_naive=pd.to_datetime(weather_raw.time_hour))[["origin", "time_hour_naive", "temp"]]
m2 = fl.merge(we_lit, on=["origin", "time_hour_naive"], how="left", validate="m:1")
joins["weather, literal join on time_hour: flights without weather temp"] = int(m2.temp.isna().sum()); assert len(m2) == n

pd.Series(joins, name="count").to_frame()

,count
airlines on carrier: flights without match,0
planes on tailnum: flights with a tail but no plane row,9371
planes on tailnum: distinct tails not in planes.csv,277
airports on dest: flights without match,0
"weather, UTC-aware join: flights without weather temp",999
"weather, literal join on time_hour: flights without weather temp",844


### 5.4 Facts that must hold

These numbers were measured on the original files when the plan was written. If one is different the notebook stops, because then something upstream changed.

In [12]:
checks = {
    "cancelled flights": (int(fl.cancelled.sum()), 10_738),
    "flights with a missing tail": (int(fl.tail_missing.sum()), 1_913),
    "missing tail and not cancelled": (int((fl.tail_missing & ~fl.is_cancelled).sum()), 0),
    "flights with a tail not in planes.csv": (int(fl.tail_not_in_planes.sum()), 9_371),
    "distinct tails not in planes.csv": (int(fl.loc[fl.tail_not_in_planes, "tailnum"].nunique()), 277),
    "departed but no arrival time": (int(fl.arrival_missing_but_departed.sum()), 715),
    "arr_time present, arr_delay missing": (int(fl.arr_delay_missing_with_arr_time.sum()), 1_081),
    "dep_time == 2400": (int((fl.dep_time == 2400).sum()), 27),
    "arr_time == 2400": (int((fl.arr_time == 2400).sum()), 198),
    "delays above 12 hours": (int(fl.overnight_delay.sum()), 365),
    "weather rows": (len(we), 26_207),
    "weather hours missing (3 airports)": (sum(len(v) for v in missing_hours.values()), 73),
    "planes: year missing": (int(pl.year.isna().sum()), 89),
    "flights without weather (UTC-aware join)": (joins["weather, UTC-aware join: flights without weather temp"], 999),
    "flights without weather (literal join)": (joins["weather, literal join on time_hour: flights without weather temp"], 844),
}
bad = {k: v for k, v in checks.items() if v[0] != v[1]}
print("all checks pass" if not bad else f"DIFFERENT: {bad}")
assert not bad
print("late rate, whole year: %.4f | train (Jan-Sep): %.4f | test (Oct-Dec): %.4f" % (
    fl.late.mean(), fl[fl.month.isin(TRAIN_MONTHS)].late.mean(), fl[fl.month.isin(TEST_MONTHS)].late.mean()))
print("cancelled rate, whole year: %.4f | train: %.4f | test: %.4f" % (
    fl.cancelled.mean(), fl[fl.month.isin(TRAIN_MONTHS)].cancelled.mean(), fl[fl.month.isin(TEST_MONTHS)].cancelled.mean()))

all checks pass
late rate, whole year: 0.2291 | train (Jan-Sep): 0.2571 | test (Oct-Dec): 0.1406
cancelled rate, whole year: 0.0247 | train: 0.0318 | test: 0.0021


### 5.5 Save the cleaned data (new files only)

In [13]:
fl.to_parquet(PROC / "flights_clean.parquet", index=False)
we.to_parquet(PROC / "weather_clean.parquet", index=False)
pl.to_parquet(PROC / "planes_clean.parquet", index=False)
ap.to_parquet(PROC / "airports_clean.parquet", index=False)
al.to_parquet(PROC / "airlines_clean.parquet", index=False)
print("saved to data/processed/:", sorted(p.name for p in PROC.glob("*_clean.parquet")))
print("flights_clean:", fl.shape)

saved to data/processed/: ['airlines_clean.parquet', 'airports_clean.parquet', 'flights_clean.parquet', 'planes_clean.parquet', 'weather_clean.parquet']
flights_clean: (435352, 35)


## 6. Data report

One line per problem: how many rows it touches, what was decided, and why.

In [14]:
c = counts
report = pd.DataFrame([
    ("Cancelled flights (dep_time and dep_delay missing)", c["cancelled flights (dep_time missing)"], "Keep + flag (is_cancelled) + target columns", "They are real events and the thing we predict; removing them would hide the cancellations."),
    ("Tail number missing", c["tailnum missing"], "Keep + flag (tail_missing). Never a model feature", "All of them are cancelled flights, so 'tail missing' would reveal the answer (leakage)."),
    ("Tail number not found in planes.csv", int(fl.tail_not_in_planes.sum()), "Keep + flag (tail_not_in_planes); plane columns filled later by airline median", "277 tails used by flights are not in the planes table; dropping them would lose 2% of flights."),
    ("Plane year missing (planes.csv)", int(pl.year.isna().sum()), "Keep; filled later by airline median in Task 5", "Only 89 planes; the plane age is a useful column otherwise."),
    ("planes.speed is 0 for every plane", len(pl), "Remove the column", "A constant column carries no information."),
    ("Departed but arrival time, delay and air time missing", c["departed but arr_time missing"], "Keep + flag (arrival_missing_but_departed)", "The data dictionary does not explain them (not called diverted); arrival is not used by the predictions."),
    ("Arrival time present but arrival delay missing", c["arr_time present but arr_delay missing"], "Keep + flag (arr_delay_missing_with_arr_time)", "Same reason; departure information is complete."),
    ("dep_time written 2400 (midnight)", c["dep_time == 2400"], "Fix in a new column dep_time_fixed", "2400 is not a valid clock time; the original is kept."),
    ("arr_time written 2400 (midnight)", c["arr_time == 2400"], "Fix in a new column arr_time_fixed", "Same."),
    ("Delays above 12 hours", c["dep_delay above 720 minutes (12 h)"], "Keep + flag (overnight_delay)", "Real flights that left the next day; the clock time has no date, so dep_delay (not dep_time) is the reliable number."),
    ("Very early departures (dep_delay below -30)", c["dep_delay below -30 minutes"], "Keep, no flag", "Possible (earliest is -50); too few rows to matter."),
    ("Implied speed below 150 or above 700 mph", c["implied speed below 150 mph"] + c["implied speed above 700 mph"], "Keep + flag (speed_flag)", "The flight happened; only air_time may be wrong, and air_time is not used by the predictions."),
    ("Duplicate rows (all files)", c["duplicate rows (flights)"] + c["duplicate rows (weather)"], "Nothing to do", "Checked: none."),
    ("Spelling or case variants in carrier, origin, dest, tailnum", sum(v for k, v in c.items() if "variants" in k), "Nothing to do", "Checked: none."),
    ("hour and minute stored as decimals, times stored as numbers HHMM", len(fl), "Fix: nullable integers; HHMM kept as given plus real date-times (sched_dt, actual_dep_dt)", "HHMM is not a number of minutes; real date-times make differences correct (including after midnight)."),
    ("time_hour stored as text, New York local time", len(fl), "Fix: real date-time with time zone, plus time_hour_utc", "Needed to join the weather correctly."),
    ("Weather time zone is UTC, flights are local", len(fl), "Fix: join on origin + UTC hour; literal join kept only as a comparison", "A literal join pairs flights with weather 4 or 5 hours off; same number of rows but wrong weather."),
    ("Weather hours missing (3 airports)", sum(len(v) for v in missing_hours.values()), "Keep missing; no interpolation of whole days", "All of Dec 31 is absent plus LGA 2023-05-02 05:00; inventing a day of weather would be a guess."),
    ("Flights without weather after the UTC-aware join", joins["weather, UTC-aware join: flights without weather temp"], "Keep; handle in Task 5/6 (NaN for trees, train median for other models)", "Mostly Dec 30-31, inside the test part (0.9% of it)."),
    ("Weather values missing: wind_dir / wind_speed and wind_gust / visib / pressure / temp", int(weather_raw.isna().any(axis=1).sum()), "Keep missing", "Single gaps; handled by the models, not by guessing."),
    ("Calm wind recorded with direction 0", int(((weather_raw.wind_speed == 0) & (weather_raw.wind_dir == 0)).sum()), "Keep + flag (wind_calm)", "Direction 0 means no wind here, not 'north'."),
    ("Target changes a lot between train and test", len(fl), "Keep; compare every model with the dumb answer on the same part", "Late rate %.1f%% in Jan-Sep against %.1f%% in Oct-Dec; cancelled %.2f%% against %.2f%%." % (
        100 * fl[fl.month.isin(TRAIN_MONTHS)].late.mean(), 100 * fl[fl.month.isin(TEST_MONTHS)].late.mean(),
        100 * fl[fl.month.isin(TRAIN_MONTHS)].cancelled.mean(), 100 * fl[fl.month.isin(TEST_MONTHS)].cancelled.mean())),
], columns=["Problem", "Rows affected", "Decision", "Why"])
report.index = report.index + 1
report

,Problem,Rows affected,Decision,Why
1,Cancelled flights (dep_time and dep_delay missing),10738,Keep + flag (is_cancelled) + target columns,They are real events and the thing we predict; removing ...
2,Tail number missing,1913,Keep + flag (tail_missing). Never a model feature,"All of them are cancelled flights, so 'tail missing' wou..."
3,Tail number not found in planes.csv,9371,Keep + flag (tail_not_in_planes); plane columns filled l...,277 tails used by flights are not in the planes table; d...
4,Plane year missing (planes.csv),89,Keep; filled later by airline median in Task 5,Only 89 planes; the plane age is a useful column otherwise.
5,planes.speed is 0 for every plane,4840,Remove the column,A constant column carries no information.
6,"Departed but arrival time, delay and air time missing",715,Keep + flag (arrival_missing_but_departed),The data dictionary does not explain them (not called di...
7,Arrival time present but arrival delay missing,1081,Keep + flag (arr_delay_missing_with_arr_time),Same reason; departure information is complete.
8,dep_time written 2400 (midnight),27,Fix in a new column dep_time_fixed,2400 is not a valid clock time; the original is kept.
9,arr_time written 2400 (midnight),198,Fix in a new column arr_time_fixed,Same.
10,Delays above 12 hours,365,Keep + flag (overnight_delay),Real flights that left the next day; the clock time has ...


## Result

* Five cleaned files saved in `data/processed/` (the originals in `data/raw/` were never touched).
* 435,352 flights kept, with the targets `late` and `cancelled`, real date-times (local and UTC), and flag columns for every problem above.
* Two traps are now documented and handled for the next tasks: the **weather file is in UTC**, and a **missing tail number always means a cancelled flight**.